# 13 — Graph vs kNN overlap (precision@K)

Subsample vertices; kNN in same feature space as 03.


### Precision@K: graph neighbors vs kNN in feature space

**Method:** Subsample vertices present in both `merged` and `G`. Build kNN in **[unit direction | log1p(Inf.Hyp.Rad)]** (same spirit as `03`). For each vertex, fraction of its **k** nearest embedding neighbors that are also **graph** neighbors defines precision@K (here K=20).

**How to read the output:** **Mean precision** near **degree/(N-1)** would be random-like for large graphs; values **clearly above** that suggest the embedding preserves local topology in this metric. It is one-sided (kNN may miss graph neighbors outside K); recall@K is a complementary metric you can add. Runtime scales with subsample size—keep `nodes` capped.


In [1]:
from pathlib import Path
import numpy as np
import networkx as nx
from sklearn.neighbors import NearestNeighbors

import dmercator3d_io as dm

merged = dm.load_merged_parquet(Path("cache/merged.parquet"))
paths = dm.paths_for_run("d3")
G = dm.load_edges_graph(paths["edge"])
name_to_i = {str(v): i for i, v in enumerate(merged["Vertex"])}
rng = np.random.default_rng(9)
nodes = [n for n in G.nodes() if str(n) in name_to_i]
if len(nodes) > 3000:
    nodes = list(rng.choice(nodes, size=3000, replace=False))
idx = np.array([name_to_i[str(n)] for n in nodes])
U = dm.normalize_direction_nd(merged)
X = np.hstack([U[idx], np.log1p(merged["Inf.Hyp.Rad"].to_numpy()[idx, None])])

K = 20
nn = NearestNeighbors(n_neighbors=K + 1).fit(X)
_, ind = nn.kneighbors(X)

def prec_at_k(i_local):
    v = nodes[i_local]
    nbr_g = set(G.neighbors(v))
    knn = [nodes[j] for j in ind[i_local, 1:]]
    hit = sum(1 for x in knn if x in nbr_g)
    return hit / K


p = np.mean([prec_at_k(i) for i in range(len(nodes))])
print("mean precision@K (graph neighbors in top-K kNN):", p)


mean precision@K (graph neighbors in top-K kNN): 0.04678333333333333
